# React 框架 · 第 10 天

## 进阶 Hooks、渲染性能、ref、Class 与 Zustand

**资料范围**：`基础拓展课 .pdf`（26 页）、当天 5 份 MD 笔记，以及 `react-pro` 对应示例代码。本文按课件主题重新组织，示例以课程使用的 **React 18.2 / Zustand 4.4** 为背景。

**本日主线**：组件内复杂状态用 `useReducer` 管理；对确有成本的渲染和计算做定点优化；通过 `ref` 提供必要的命令式能力；读懂 Class 组件；把跨组件状态放到 Zustand store 中。

## 一、学习目标与知识地图

完成本日后应能：

1. 写出 `reducer(state, action)`，理解 `dispatch`、`type`、`payload` 和状态更新流程。
2. 分清 `useMemo` 缓存**计算结果**、`memo` 跳过**子组件渲染**、`useCallback` 稳定**函数引用**。
3. 用 `forwardRef` 转发 ref，并用 `useImperativeHandle` 只暴露必要方法。
4. 看懂 Class 组件的 `state`、`setState`、`render`、挂载与卸载生命周期、父子通信。
5. 用 Zustand 创建 store、读取选定状态、执行同步或异步 action，并拆分大型 store。

```text
状态管理：组件内 useState / useReducer ──跨组件──> Zustand
性能优化：先发现多余计算或渲染 ──再选──> useMemo / memo / useCallback
特殊交互：声明式 props 优先 ──确需聚焦等命令──> ref
```

## 二、`useReducer`：把状态变化写成规则

当多个事件围绕同一份状态变化、更新规则逐渐复杂时，把规则集中到 reducer 中更容易维护。

```jsx
import { useReducer } from 'react'

function counterReducer(state, action) {
  switch (action.type) {
    case 'increment':
      return state + 1
    case 'decrement':
      return state - 1
    case 'set':
      return action.payload
    default:
      return state
  }
}

function Counter() {
  const [count, dispatch] = useReducer(counterReducer, 0)

  return (
    <div>
      <button onClick={() => dispatch({ type: 'decrement' })}>−</button>
      <span>{count}</span>
      <button onClick={() => dispatch({ type: 'increment' })}>＋</button>
      <button onClick={() => dispatch({ type: 'set', payload: 100 })}>
        设为 100
      </button>
    </div>
  )
}
```

`dispatch(action)` 发起一次更新 → React 将当前状态和 action 交给 reducer → reducer 返回**下一个状态** → React 据此更新界面。`payload` 只是约定俗成的字段名，可以存放本次操作的数据。

![课程资料中的 useReducer 更新流程](assets/day10/01.png)

**资料勘误**：原例写成 `state + action.payload`，按钮却叫“update to 100”。那是“增加 100”，不是“设为 100”；上例使用 `return action.payload` 与按钮语义一致。

## 三、复杂状态、不可变更新与 `useState` 的选择

reducer 不限于数字。表单有多个关联字段时，可以把变化集中描述：

```jsx
const initialState = { title: '', status: 'draft' }

function formReducer(state, action) {
  switch (action.type) {
    case 'titleChanged':
      return { ...state, title: action.payload }
    case 'published':
      return { ...state, status: 'published' }
    case 'reset':
      return initialState
    default:
      return state
  }
}
```

`state` 和 `action` 应视为只读输入。对象或数组更新时返回新对象、新数组，不要直接修改旧状态。reducer 应保持纯粹：不要在里面发请求、写浏览器存储或触发其他副作用。

| 场景 | 建议 |
|---|---|
| 一个简单开关、输入框或计数器 | `useState` 足够直接 |
| 多种事件作用于同一份复杂状态，更新规则相互关联 | `useReducer` 更清楚 |
| 多个不相邻组件共享同一份状态 | 结合 Context，或考虑 Zustand 等共享状态方案 |

`useReducer` 仍是**当前组件的状态管理方式**；单独使用它不会自动让全应用共享状态。

## 四、先理解渲染，再谈优化

当组件状态更新时，React 会重新调用该组件函数以计算新 UI；其子组件通常也会随之参与渲染。**重新渲染不等于整页 DOM 被重建**。优化的对象是实际出现的昂贵计算、昂贵子树或不必要的更新。

本日三个工具解决不同问题：

| 工具 | 缓存或跳过什么 | 典型触发条件 |
|---|---|---|
| `useMemo(calculate, deps)` | 计算结果 | 依赖变化时重新计算 |
| `memo(Component)` | props 未变时跳过子组件渲染 | 任一 prop 变化时重新渲染 |
| `useCallback(fn, deps)` | 函数引用 | 依赖变化时得到新函数 |

先用日志或 React DevTools 观察，再优化可感知的瓶颈。每种缓存也有维护依赖和比较的成本，不必给每个变量或函数都加一层。

## 五、`useMemo`：依赖不变时复用计算结果

若 `count` 决定一项较慢的计算，而另一个状态 `num` 频繁变化，可以让这项计算只随 `count` 改变：

```jsx
import { useMemo, useState } from 'react'

function fibonacci(n) {
  if (n <= 1) return n
  return fibonacci(n - 1) + fibonacci(n - 2)
}

function App() {
  const [count, setCount] = useState(10)
  const [num, setNum] = useState(0)

  const result = useMemo(() => fibonacci(count), [count])

  return (
    <>
      <p>F({count}) = {result}</p>
      <button onClick={() => setCount(c => c + 1)}>计算下一个</button>
      <button onClick={() => setNum(n => n + 1)}>其他状态：{num}</button>
    </>
  )
}
```

点击“其他状态”会使组件重新渲染，但 `count` 未变时，可复用上次结果；改变 `count` 则重新计算。示例是为展示机制，递归 Fibonacci 在数值大时本身很慢，实际业务应采用更合适的算法。

**资料勘误**：课件第一段把“阶乘”的递归公式称为“斐波那契”。上面统一使用真正的 Fibonacci 公式，避免概念混淆。

## 六、`React.memo`：props 稳定时跳过子组件渲染

```jsx
import { memo, useState } from 'react'

const Child = memo(function Child({ label }) {
  console.log('Child render')
  return <p>{label}</p>
})

function Parent() {
  const [other, setOther] = useState(0)
  return (
    <>
      <Child label="固定内容" />
      <button onClick={() => setOther(n => n + 1)}>{other}</button>
    </>
  )
}
```

父组件因 `other` 更新而重新渲染，但 `label` 不变时，`memo` 通常可让 `Child` 跳过这次渲染。默认比较是对每个 prop 做浅层 `Object.is` 比较。

```jsx
<Child items={[1, 2, 3]} /> // 每次父组件渲染都会产生新数组引用
```

对象、数组、函数即使“内容看起来一样”，新引用也会被视为变化。需要时可让引用稳定，或为 `memo` 提供自定义比较函数；后者要覆盖**所有影响渲染的 props**，且比较本身不能比重渲染更贵。子组件自身 state 或所用 Context 变化时，`memo` 也不能阻止相应渲染。

## 七、`useCallback`：给回调稳定的引用

回调作为 prop 传给 `memo` 子组件时，父组件每次渲染都新建函数，会使浅比较失效。此时可使用 `useCallback`：

```jsx
import { memo, useCallback, useState } from 'react'

const Child = memo(function Child({ onReport }) {
  console.log('Child render')
  return <button onClick={() => onReport('来自子组件')}>报告</button>
})

function Parent() {
  const [other, setOther] = useState(0)
  const onReport = useCallback(message => {
    console.log(message)
  }, [])

  return (
    <>
      <Child onReport={onReport} />
      <button onClick={() => setOther(n => n + 1)}>{other}</button>
    </>
  )
}
```

这里 `onReport` 不读取变化中的状态，因此空依赖数组成立。如果回调读取 `count` 等变化值，应将其列入依赖，或使用函数式更新来减少依赖。漏填依赖会让回调使用过期值。

**组合关系**：`memo` 判断子组件要不要重渲染；`useCallback` 帮传给它的函数 prop 保持引用稳定；`useMemo` 可保持昂贵计算的结果或对象 prop 稳定。只有出现对应问题时再组合使用。

## 八、`forwardRef`：让父组件拿到子组件内的 DOM

普通函数组件的 `ref` 在本课程 React 18 写法中不会自动转发给内部 `<input>`；需要 `forwardRef`：

```jsx
import { forwardRef, useRef } from 'react'

const MyInput = forwardRef(function MyInput(props, ref) {
  return <input {...props} ref={ref} />
})

function App() {
  const inputRef = useRef(null)
  return (
    <>
      <MyInput ref={inputRef} placeholder="请输入" />
      <button onClick={() => inputRef.current?.focus()}>聚焦输入框</button>
    </>
  )
}
```

`forwardRef` 的第二个参数是转发而来的 ref；上例把它交给真实 DOM。访问 `.current` 要考虑组件尚未挂载的情况。课程原例在 `forwardRef(..., [])` 末尾多写了一个数组参数，实际调用不需要它。

## 九、`useImperativeHandle`：只暴露有限的命令式能力

如果父组件只需“聚焦”功能，无需拿到整个 DOM 节点，可以给 ref 暴露一个小接口：

```jsx
import { forwardRef, useImperativeHandle, useRef } from 'react'

const MyInput = forwardRef(function MyInput(props, ref) {
  const innerRef = useRef(null)

  useImperativeHandle(ref, () => ({
    focus() {
      innerRef.current?.focus()
    },
  }), [])

  return <input {...props} ref={innerRef} />
})

function App() {
  const inputApiRef = useRef(null)
  return (
    <>
      <MyInput ref={inputApiRef} />
      <button onClick={() => inputApiRef.current?.focus()}>聚焦</button>
    </>
  )
}
```

适用场景包括聚焦、滚动、播放/暂停等难以仅靠 props 表达的操作。常规数据流仍用 props 和状态；不要让父组件通过 ref 随意修改子组件内部状态。

## 十、Class 组件：结构、状态与事件

课程后半段介绍旧项目中仍可能遇到的 Class API。组件继承 `Component`，在 `render()` 中返回 JSX，使用 `this.state` 保存状态、`this.setState` 更新状态：

```jsx
import { Component } from 'react'

class Counter extends Component {
  state = { count: 0 }

  increment = () => {
    this.setState(prevState => ({ count: prevState.count + 1 }))
  }

  render() {
    return <button onClick={this.increment}>{this.state.count}</button>
  }
}
```

`setState` 通知 React 更新并安排重新渲染；不要直接给 `this.state.count` 赋值。当新值依赖旧值时使用函数式更新，以免连续更新读到过期状态。类字段箭头函数可保持事件回调中的 `this` 指向组件实例。

## 十一、Class 生命周期与函数组件对照

```jsx
class Clock extends Component {
  state = { now: new Date() }

  componentDidMount() {
    this.timer = setInterval(() => {
      this.setState({ now: new Date() })
    }, 1000)
  }

  componentWillUnmount() {
    clearInterval(this.timer)
  }

  render() {
    return <time>{this.state.now.toLocaleTimeString()}</time>
  }
}
```

- `render()`：计算界面，不应在其中启动定时器或请求。
- `componentDidMount()`：首次挂载后，可建立订阅、定时器或发起加载。
- `componentWillUnmount()`：卸载前清理订阅、定时器等资源。
- 更新阶段还存在 `componentDidUpdate()`，可响应 props/state 变化，但必须防止无条件 `setState` 导致循环。

函数组件常用 `useEffect` 表达“建立副作用并返回清理函数”。两种模型可对照理解，但并不是把每个生命周期方法机械地翻译成一个 Hook。

## 十二、Class 组件通信：props 向下，回调向上

```jsx
class Child extends Component {
  render() {
    const { message, onMessage } = this.props
    return (
      <button onClick={() => onMessage('来自子组件')}>
        {message}
      </button>
    )
  }
}

class Parent extends Component {
  state = { message: '初始消息' }

  handleMessage = message => this.setState({ message })

  render() {
    return (
      <Child
        message={this.state.message}
        onMessage={this.handleMessage}
      />
    )
  }
}
```

父传子：`Parent` 通过 props 传 `message`；子传父：`Parent` 把回调作为 prop 传下去，`Child` 在事件中调用。状态所有权仍在父组件。这与函数组件的通信思路相同，差别主要是 Class 中通过 `this.props`、`this.state` 访问。

## 十三、Zustand：创建与读取共享 store

Zustand 将共享状态与更新函数放在 store 中。本课程代码使用 `create` 创建一个可在组件中调用的 Hook：

```js
// store/useCounterStore.js
import { create } from 'zustand'

export const useCounterStore = create(set => ({
  count: 0,
  increment: () => set(state => ({ count: state.count + 1 })),
}))
```

```jsx
import { useCounterStore } from './store/useCounterStore'

function Counter() {
  const count = useCounterStore(state => state.count)
  const increment = useCounterStore(state => state.increment)
  return <button onClick={increment}>{count}</button>
}
```

`set(state => partialState)` 适合依赖旧值的更新；在此课程版本中，部分对象会合并到 store。组件通过 selector 只订阅自己需要的值。课程 MD 中 `const { count, inc } = useStore()` 也能工作，但读取整个 store 会使该组件更容易受无关字段更新影响。

## 十四、Zustand 的异步 action

异步逻辑可直接写在 store 的 action 中，取得结果后再调用 `set`：

```js
import { create } from 'zustand'

const URL = 'http://geek.itheima.net/v1_0/channels'

export const useChannelStore = create(set => ({
  channels: [],
  loading: false,
  error: null,

  fetchChannels: async () => {
    set({ loading: true, error: null })
    try {
      const response = await fetch(URL)
      if (!response.ok) throw new Error(`HTTP ${response.status}`)
      const json = await response.json()
      set({ channels: json.data.channels, loading: false })
    } catch (error) {
      set({ error: error.message, loading: false })
    }
  },
}))
```

组件中使用 `useEffect(() => { fetchChannels() }, [fetchChannels])` 在挂载时触发；页面展示 `loading`、`error`、列表三种状态。课程最简例只展示成功路径；这里补上失败与加载状态，便于实际使用。示例地址来自课件，运行效果取决于该服务是否可访问。

## 十五、Zustand 切片：按领域拆分大 store

当一个 store 同时承载计数器、频道等不同领域，可把状态和 action 分成函数后组合：

```js
import { create } from 'zustand'

const createCounterSlice = set => ({
  count: 0,
  increment: () => set(state => ({ count: state.count + 1 })),
})

const createChannelSlice = set => ({
  channels: [],
  setChannels: channels => set({ channels }),
})

export const useAppStore = create((set, get, api) => ({
  ...createCounterSlice(set, get, api),
  ...createChannelSlice(set, get, api),
}))
```

`count` 与 `channels` 在同一个 store 中，但实现按领域分开，组件仍可用 selector 精确读取。实际项目要避免切片间字段名或 action 名冲突。

**资料勘误**：课程切片示例定义了 `setCount`、`fetchGetList`，组件却调用 `inc`、`fetchChannelList`；照抄会报错。本例统一使用 `increment`、`setChannels`。

## 十六、调试与工具选择

课程 MD 还介绍 `simple-zustand-devtools`，可在开发环境观察 store 状态；项目若使用它，应按安装版本的说明配置，并确保调试挂载逻辑只在开发环境执行。React DevTools 可辅助观察组件和渲染。

![课程资料中的 Zustand 调试示例](assets/day10/02.png)

| 问题 | 优先考虑 |
|---|---|
| 单个组件的简单状态 | `useState` |
| 单个组件内多种关联状态转移 | `useReducer` |
| 非相邻组件频繁共享、修改同一状态 | Zustand store |
| 一次昂贵计算因无关状态反复执行 | `useMemo` |
| 重型子组件因父组件更新而重复渲染 | `memo`，必要时配合稳定的 props |
| 父组件需要控制子组件 DOM 的聚焦等行为 | `forwardRef` / `useImperativeHandle` |

不应仅凭“会重新渲染”就加缓存。先确认问题确实存在，再选最小的手段解决。

## 十七、易错点速查

1. **把 reducer 当成副作用函数**：reducer 只计算新状态；请求、定时器等放到事件处理、Effect 或 store action。
2. **直接改对象状态**：`state.title = 'x'` 会破坏不可变更新约定；返回新对象。
3. **`useMemo` 依赖漏填**：计算中用到的变化值必须放入依赖，否则可能得到过期结果。
4. **以为 `memo` 会深比较**：默认按 prop 做浅层 `Object.is`；新对象/数组/函数引用会使比较不通过。
5. **给所有函数加 `useCallback`**：它本身不会消除计算或让父组件停止渲染；先确认接收方是否依赖稳定引用。
6. **把 ref 当普通数据通道**：用 props/state 传数据；ref 适合少量命令式交互。
7. **Class 中直接修改 `this.state`**：应调用 `this.setState`；依赖旧值时用函数式写法。
8. **订阅整个 Zustand store**：无关状态更新也可能带动组件；按需使用 selector。
9. **切片方法命名不一致**：定义、导出和调用必须同名，课件相关示例有笔误。

## 十八、动手练习

1. **状态规则**：为购物车写 reducer，支持 `add`、`remove`、`changeQuantity`、`clear`，每个 action 含所需 `payload`。检查原状态对象未被直接改动。
2. **性能观察**：父组件维护两个独立状态；子组件接收一个列表和一个回调。先用日志观察重渲染，再分别加入 `memo`、`useMemo`、`useCallback`，解释每一步是否真正减少渲染。
3. **命令式输入框**：封装 `SearchInput`，只向父组件暴露 `focus()` 与 `clear()`，并用两个按钮验证。
4. **Class 读写**：把函数版计数器改写为 Class；添加每秒更新的时钟并在卸载时清理定时器。
5. **共享频道列表**：用 Zustand 保存列表、加载中和错误状态，两个组件分别读取列表数量和列表内容；观察 selector 对更新范围的影响。

自测时重点解释**为什么**选某个 API，而不是只记住调用形式。

## 十九、本日小结

`useReducer` 用 action 描述组件内关联状态的变化；`useMemo`、`memo`、`useCallback` 分别处理重复计算、子组件重复渲染和回调引用变化。`forwardRef` 与 `useImperativeHandle` 为聚焦等少量命令式操作提供受控入口。Class API 帮助阅读旧代码，核心仍是状态驱动界面、props 向下和回调向上。Zustand 把共享状态抽到 store，异步 action 和切片让它随业务规模继续扩展。

**记忆顺序**：先保证数据流和状态更新正确，再观察瓶颈，最后做有证据的优化。